# Experimento: Comparação GBML vs River vs ACDWM

**Propósito**: Executar comparação completa dos 3 tipos de modelos

**Autor**: Claude Code  
**Data**: 2025-01-07

---

## Pré-requisitos:

1. ✓ Executar **setup_acdwm_colab.ipynb** primeiro
2. ✓ Google Drive montado
3. ✓ Dependências instaladas
4. ✓ ACDWM clonado

---

## Estratégia de Execução:

### Fase 1: Teste Rápido (RECOMENDADO)
- 1 dataset: RBF_Abrupt_Severe
- 2 chunks (chunk_size=1000)
- Modelos: GBML + HAT + ACDWM
- Tempo estimado: ~5-10 minutos

### Fase 2: Experimento Completo
- 3 datasets
- 3 chunks por dataset (chunk_size=6000)
- Modelos: GBML + HAT + ARF + SRP + ACDWM
- Tempo estimado: ~10-15 horas


---
## 1. Setup Inicial


In [ ]:
from google.colab import drive
import os
import sys

# Monta Drive
drive.mount('/content/drive')

# ========================================
# AJUSTE ESTE CAMINHO!
# ========================================
DRIVE_PATH = '/content/drive/MyDrive/DSL-AG-hybrid'

# Configuracao
os.chdir(DRIVE_PATH)
sys.path.insert(0, DRIVE_PATH)

print(f"[OK] Working directory: {os.getcwd()}")
print(f"[OK] ACDWM path: {os.path.join(DRIVE_PATH, 'ACDWM')}")

---
## 2. TESTE RÁPIDO: 1 Dataset, 2 Chunks

**COMECE POR AQUI** para validar que tudo funciona!


In [ ]:
%%time

print("="*70)
print("TESTE RAPIDO: 1 Dataset, 2 Chunks")
print("Modelos: GBML + HAT + ACDWM")
print("="*70)

!python compare_gbml_vs_river.py \
    --stream RBF_Abrupt_Severe \
    --config config_comparison.yaml \
    --models HAT \
    --chunks 2 \
    --chunk-size 1000 \
    --acdwm \
    --seed 42 \
    --output test_quick_results

print("\n" + "="*70)
print("[OK] TESTE RAPIDO CONCLUIDO!")
print("="*70)

---
## 3. Verificar Resultados do Teste Rápido


In [ ]:
import pandas as pd
import glob

# Busca arquivo de resultados
result_files = glob.glob('test_quick_results/**/comparison_table.csv', recursive=True)

if result_files:
    print(f"[OK] Arquivo de resultados encontrado: {result_files[0]}\n")
    
    # Le resultados
    df = pd.read_csv(result_files[0])
    
    print("Resultados:")
    print("="*70)
    print(df[['chunk_idx', 'model', 'accuracy', 'gmean', 'f1_weighted']].to_string(index=False))
    
    print("\n" + "="*70)
    print("Medias por modelo:")
    print("="*70)
    print(df.groupby('model')[['accuracy', 'gmean', 'f1_weighted']].mean().round(4))
    
else:
    print("[X] Nenhum arquivo de resultados encontrado!")
    print("\nArquivos em test_quick_results:")
    !ls -lR test_quick_results/

---
## 4. EXPERIMENTO INTERMEDIÁRIO: 1 Dataset, 3 Chunks

Teste com configuração próxima do experimento final.


In [ ]:
%%time

print("="*70)
print("EXPERIMENTO INTERMEDIARIO: 1 Dataset, 3 Chunks")
print("Modelos: GBML + HAT + ARF + ACDWM")
print("="*70)

!python compare_gbml_vs_river.py \
    --stream RBF_Abrupt_Severe \
    --config config_comparison.yaml \
    --models HAT ARF \
    --chunks 3 \
    --chunk-size 6000 \
    --acdwm \
    --seed 42 \
    --output test_intermediate_results

print("\n" + "="*70)
print("[OK] EXPERIMENTO INTERMEDIARIO CONCLUIDO!")
print("="*70)

---
## 5. Verificar Resultados Intermediários


In [ ]:
import pandas as pd
import glob

result_files = glob.glob('test_intermediate_results/**/comparison_table.csv', recursive=True)

if result_files:
    print(f"[OK] Resultados encontrados: {result_files[0]}\n")
    
    df = pd.read_csv(result_files[0])
    
    print("Resultados por chunk:")
    print("="*70)
    print(df[['chunk_idx', 'model', 'accuracy', 'gmean', 'f1_weighted']].to_string(index=False))
    
    print("\n" + "="*70)
    print("Estatisticas:")
    print("="*70)
    summary = df.groupby('model')[['accuracy', 'gmean', 'f1_weighted']].agg(['mean', 'std'])
    print(summary.round(4))
    
    # Melhor modelo por metrica
    print("\n" + "="*70)
    print("Melhor modelo por metrica (media):")
    print("="*70)
    means = df.groupby('model')[['accuracy', 'gmean', 'f1_weighted']].mean()
    for metric in ['accuracy', 'gmean', 'f1_weighted']:
        best_model = means[metric].idxmax()
        best_value = means[metric].max()
        print(f"  {metric:15s}: {best_model:10s} ({best_value:.4f})")
else:
    print("[X] Nenhum resultado encontrado!")

---
## 6. EXPERIMENTO COMPLETO: 3 Datasets

⚠️ **ATENÇÃO**: Isso pode levar 10-15 horas!

Execute apenas se os testes anteriores funcionaram corretamente.


In [ ]:
%%time

print("="*70)
print("EXPERIMENTO COMPLETO")
print("3 Datasets x 3 Chunks x 3 Modelos River")
print("TEMPO ESTIMADO: 10-15 HORAS")
print("="*70)

from datetime import datetime
start_time = datetime.now()
print(f"\nInicio: {start_time.strftime('%Y-%m-%d %H:%M:%S')}\n")

!python run_comparison_colab.py

end_time = datetime.now()
duration = (end_time - start_time).total_seconds() / 3600

print("\n" + "="*70)
print("[OK] EXPERIMENTO COMPLETO CONCLUIDO!")
print("="*70)
print(f"Inicio:   {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Fim:      {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Duracao:  {duration:.2f} horas")

---
## 7. Analisar Resultados Completos


In [ ]:
import pandas as pd
import glob

# Busca resultados consolidados
consolidated_file = glob.glob('comparison_results/**/consolidated_results.csv', recursive=True)

if consolidated_file:
    print(f"[OK] Resultados consolidados: {consolidated_file[0]}\n")
    
    df = pd.read_csv(consolidated_file[0])
    
    print("Datasets incluidos:")
    print(df['dataset'].unique())
    
    print("\nModelos incluidos:")
    print(df['model'].unique())
    
    print("\n" + "="*70)
    print("RESULTADOS CONSOLIDADOS")
    print("="*70)
    
    # Agrupa por dataset e modelo
    summary = df.groupby(['dataset', 'model'])[['accuracy', 'gmean', 'f1_weighted']].agg(['mean', 'std'])
    print(summary.round(4))
    
    # Melhor modelo por dataset
    print("\n" + "="*70)
    print("MELHOR MODELO POR DATASET (G-mean)")
    print("="*70)
    
    for dataset in df['dataset'].unique():
        df_dataset = df[df['dataset'] == dataset]
        means = df_dataset.groupby('model')['gmean'].mean()
        best_model = means.idxmax()
        best_value = means.max()
        print(f"  {dataset:30s}: {best_model:10s} (G-mean={best_value:.4f})")
    
    # Ranking geral
    print("\n" + "="*70)
    print("RANKING GERAL (media de G-mean em todos os datasets)")
    print("="*70)
    
    overall = df.groupby('model')['gmean'].mean().sort_values(ascending=False)
    for i, (model, gmean) in enumerate(overall.items(), 1):
        print(f"  {i}. {model:10s}: {gmean:.4f}")
    
else:
    print("[X] Resultados consolidados nao encontrados!")
    print("\nArquivos disponiveis:")
    !find comparison_results -name "*.csv" -type f

---
## 8. Gerar Gráficos de Comparação


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob

consolidated_file = glob.glob('comparison_results/**/consolidated_results.csv', recursive=True)

if consolidated_file:
    df = pd.read_csv(consolidated_file[0])
    
    # Configuracao de estilo
    sns.set_style('whitegrid')
    plt.rcParams['figure.figsize'] = (14, 8)
    
    # Grafico 1: G-mean por dataset e modelo
    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    
    summary = df.groupby(['dataset', 'model'])['gmean'].mean().reset_index()
    
    sns.barplot(data=summary, x='dataset', y='gmean', hue='model', ax=ax)
    ax.set_title('G-mean por Dataset e Modelo', fontsize=16, fontweight='bold')
    ax.set_xlabel('Dataset', fontsize=12)
    ax.set_ylabel('G-mean (media)', fontsize=12)
    ax.legend(title='Modelo', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.set_ylim(0, 1)
    
    plt.tight_layout()
    plt.show()
    
    # Grafico 2: Boxplot de G-mean por modelo
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    
    sns.boxplot(data=df, x='model', y='gmean', ax=ax)
    ax.set_title('Distribuicao de G-mean por Modelo', fontsize=16, fontweight='bold')
    ax.set_xlabel('Modelo', fontsize=12)
    ax.set_ylabel('G-mean', fontsize=12)
    ax.set_ylim(0, 1)
    
    plt.tight_layout()
    plt.show()
    
    print("[OK] Graficos gerados!")
else:
    print("[X] Sem resultados para plotar")

---
## 9. Salvar Resultados no Drive


In [ ]:
import shutil
import os
from datetime import datetime

# Cria pasta de backup com timestamp
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
backup_dir = f"comparison_results_backup_{timestamp}"

if os.path.exists('comparison_results'):
    print(f"Copiando resultados para: {backup_dir}")
    shutil.copytree('comparison_results', backup_dir)
    print(f"\n[OK] Backup salvo em: {os.path.join(DRIVE_PATH, backup_dir)}")
    print("\nOs resultados foram salvos no Google Drive!")
else:
    print("[X] Pasta comparison_results nao encontrada")

---
## Resumo Final

### Arquivos Gerados:

```
comparison_results/
├── experiment_TIMESTAMP/
│   ├── experiment_config.json
│   ├── experiment_summary.json
│   ├── consolidated_results.csv
│   ├── summary_statistics.txt
│   └── [dataset]_seed42/
│       ├── comparison_table.csv
│       ├── GBML_results.json
│       └── River_[model]_results.json
```

### Principais Métricas:

- **Accuracy**: Precisão geral
- **G-mean**: Média geométrica (importante para dados desbalanceados)
- **F1-weighted**: F1 ponderado por classe
- **Precision/Recall**: Por classe

### Próximos Passos:

1. Analisar resultados consolidados
2. Identificar melhor modelo por dataset
3. Comparar tempo de execução
4. Escrever relatório final
